# Notebook Analyse Textsemantic (ERP prétraités)

Ce notebook reprend l'analyse des potentiels évoqués issus du paradigme de lecture de phrases (texte congruent vs incongruent).
Comme dans `01_preprocessing_notebook.ipynb` et `02_analysis1_gonogo.ipynb`, chaque cellule est balisée :
- **Type 1 — prêt à exécuter** : code fonctionnel, vous pouvez simplement l’exécuter pour reproduire l’analyse.
- **Type 2 — à personnaliser** : zones où modifier des paramètres (balises `A_COMPLETER`, valeurs exemples, etc.).
- **Type 3 — exploration** : pistes pour approfondir ou tester d’autres idées (conditions, fenêtres temporelles, métriques supplémentaires).

## Pour bien démarrer
- Placez/montez le dossier `tasks/text_reading/bids` au même niveau que ce carnet, ou ajustez `CANDIDATE_ROOTS`.
- Assurez-vous d’avoir exécuté le pipeline de prétraitement afin de disposer des fichiers `*_processed.fif` dans `derivatives/preproc`.
- Activez l’environnement Python du cours et installez les dépendances (`pip install -r requirements.txt`).

## Objectifs pédagogiques
1. Charger un enregistrement textsemantic déjà prétraité et vérifier ses métadonnées.
2. Extraire les événements congruent / incongruent, créer des epochs équilibrés et visualiser les ERP.
3. Mesurer la N400 (amplitude/latence) au niveau sujet et au niveau groupe, en sauvegardant les résultats pour réutilisation ultérieure.


## 0. Préparation et configuration

Nous configurons l’environnement d’analyse : imports, chemins BIDS, sélection du sujet. Chaque bloc détaille ce qu’il fait et comment vous pouvez l’adapter pour vos explorations (ex. utiliser un autre chemin, ajouter un sujet, jouer avec les paramètres de rejet).


### Bloc Type 1 — Imports et options globales

Charge les bibliothèques principales (MNE, NumPy, Matplotlib, outils BIDS) et fixe quelques réglages d’affichage. Si vous expérimentez d’autres packages (par ex. `seaborn` ou `scipy`), importez-les ici pour garder le carnet cohérent.


In [ ]:
# -----------------------------------------------------------------------------
# Imports principaux et configuration globale
# -----------------------------------------------------------------------------
import warnings  # contrôle des avertissements Python
from pathlib import Path  # manipulation de chemins indépendante de l'OS
import csv  # lecture / écriture de fichiers tabulés (participants.tsv)
import json  # sauvegarde des métriques intermédiaires
from collections import Counter  # comptage rapide des annotations

import matplotlib.pyplot as plt  # visualisation des ERP
import mne  # bibliothèque principale pour l'analyse EEG/MEG
import numpy as np  # calcul scientifique vectorisé
import mne_bids  # accès à la version et aux outils BIDS
from mne_bids import BIDSPath  # gestion de chemins compatibles BIDS

warnings.filterwarnings('ignore', category=RuntimeWarning)  # masque certains warnings MNE
mne.set_log_level('INFO')  # verbosité modérée pour suivre les étapes clés
plt.rcParams['figure.figsize'] = (10, 5)  # taille par défaut des figures

print('Versions utilisées:')
print(' - mne      ', mne.__version__)
print(' - numpy    ', np.__version__)
print(' - mne_bids ', mne_bids.__version__)


### Bloc Type 1 — Définir le dossier BIDS et les dérivés

Localise les données brutes et configure les dossiers de sortie :
- `derivatives/preproc` contient les fichiers nettoyés produits par le notebook 01 (AutoReject + ICA).
- `derivatives/textsemantic-erp` regroupe les ERP de groupe fournis avec le dataset (optionnel).
- `derivatives/textsemantic-analysis` accueille les artefacts générés ici (métriques JSON, difference waves, exports CSV). Vous pouvez créer d’autres sous-dossiers si vous développez de nouvelles analyses.


In [ ]:
# -----------------------------------------------------------------------------
# Localisation des données BIDS et des dérivés nécessaires
# -----------------------------------------------------------------------------
CANDIDATE_ROOTS = [
    Path('../../data/text_reading/bids'),
]

root_bids = next((candidate for candidate in CANDIDATE_ROOTS if candidate.exists()), None)
if root_bids is None:
    raise FileNotFoundError('Aucun dossier BIDS détecté — ajoutez votre chemin dans CANDIDATE_ROOTS.')

# Dérivés harmonisés avec le notebook de prétraitement
deriv_preproc = root_bids / 'derivatives' / 'preproc'
deriv_preproc.mkdir(parents=True, exist_ok=True)

# Dérivés spécifiques à cette analyse
deriv_erp = root_bids / 'derivatives' / 'textsemantic-erp'
deriv_analysis = root_bids / 'derivatives' / 'textsemantic-analysis'
deriv_analysis.mkdir(parents=True, exist_ok=True)

TASK_LABEL = 'textsemantic'
FOCUS_CONDITIONS = ['cw_cong', 'cw_incong']

print('Chemins vérifiés:')
print(' - BIDS root            :', root_bids.resolve())
print(' - dérivés préproc      :', deriv_preproc.resolve())
print(' - dérivés analyse      :', deriv_analysis.resolve())
print(' - dérivés ERP (option) :', deriv_erp.resolve())


### Bloc Type 1 — Lister les participants disponibles

Lit `participants.tsv` pour récupérer les identifiants `sub-XX`. Si vous ajoutez / retirez des sujets, relancez cette cellule pour mettre à jour la liste. Vous pouvez également filtrer cette liste (ex. ne garder que certains sujets) avant de passer à la suite.


In [ ]:
# -----------------------------------------------------------------------------
# Lecture de participants.tsv pour récupérer la liste des sujets
# -----------------------------------------------------------------------------
participants_tsv = root_bids / 'participants.tsv'
if not participants_tsv.exists():
    raise FileNotFoundError(f'participants.tsv introuvable: {participants_tsv}')

subjects = []
with participants_tsv.open('r', encoding='utf-8') as f:
    reader = csv.reader(f, delimiter='	')
    header = next(reader, None)  # ignore l'en-tête
    for row in reader:
        if not row:
            continue
        participant_id = row[0]
        if participant_id.startswith('sub-'):
            subjects.append(participant_id.replace('sub-', ''))

if not subjects:
    raise RuntimeError('Aucun sujet détecté dans participants.tsv')

print('Sujets disponibles :', subjects)


### Bloc Type 2 — Sélectionner un participant et une session

Choisissez le sujet/session/run à analyser. Pour explorer d’autres données, modifiez ces variables puis relancez les cellules suivantes. Vous pouvez automatiser cela (ex. boucle sur plusieurs sujets) en vous inspirant de la section multi-sujets en fin de carnet.


In [ ]:
# -----------------------------------------------------------------------------
# Choix du sujet / session / run à analyser
# -----------------------------------------------------------------------------
subject = '10'  # <-- remplacez par un identifiant présent dans `subjects`
session = '001'  # la plupart des sujets possèdent cette session unique
run = '01'  # un seul run textsemantic est disponible

print(f'Sujet en cours: sub-{subject}, session {session}, run {run}')


## 1. Charger un enregistrement prétraité

Nous ouvrons le fichier `*_processed.fif` correspondant au sujet sélectionné, afin de partir des données déjà nettoyées (filtrage, AutoReject, ICA).


### Bloc Type 1 — Fonction utilitaire de chargement

`load_processed_raw` construit un `BIDSPath` pointant vers `derivatives/preproc`, charge le Raw prétraité et vérifie que le montage EEG est défini. Réutilisez cette fonction si vous écrivez d’autres scripts d’analyse sur les mêmes données.


In [ ]:
# -----------------------------------------------------------------------------
# Fonction utilitaire : charge le fichier *_processed.fif depuis derivatives/preproc
# -----------------------------------------------------------------------------
def load_processed_raw(subject: str, session: str = '001', run: str = '01') -> tuple[mne.io.BaseRaw, Path]:
    processed_bids = BIDSPath(
        root=deriv_preproc,
        subject=subject,
        session=session,
        task=TASK_LABEL,
        run=run,
        datatype='eeg',
        processing='clean',
        suffix='processed',
        extension='.fif'
    )
    if not processed_bids.fpath.exists():
        raise FileNotFoundError(f'Fichier prétraité introuvable: {processed_bids.fpath}')

    raw_obj = mne.io.read_raw_fif(processed_bids.fpath, preload=True)
    if raw_obj.get_montage() is None:
        raw_obj.set_montage('standard_1020', match_case=False, on_missing='warn')
    return raw_obj, processed_bids.fpath


### Bloc Type 1 — Charger et inspecter les métadonnées

Contrôle rapide : fréquence d’échantillonnage, liste des canaux, annotations disponibles. Comparez les annotations au cahier de tâches : cela permet de confirmer que les codes congruent/incongruent sont présents avant de découper en epochs.


In [ ]:
# -----------------------------------------------------------------------------
# Chargement du fichier sélectionné et résumé des métadonnées
# -----------------------------------------------------------------------------
raw, processed_path = load_processed_raw(subject, session=session, run=run)
print('Fichier prétraité chargé :', processed_path)


In [ ]:
# -----------------------------------------------------------------------------
# Inspection rapide : fréquence, canaux, annotations
# -----------------------------------------------------------------------------
print("Fréquence d'échantillonnage :", raw.info['sfreq'], "Hz")
print("Canaux EEG                :", len(mne.pick_types(raw.info, eeg=True)))
print("Canaux bad                :", raw.info['bads'])
print("Durée totale (minutes)    :", raw.times[-1] / 60.0)

annotation_counts = Counter(str(desc) for desc in raw.annotations.description)
print('Annotations disponibles   :')
for desc, count in annotation_counts.items():
    print(f' - {desc}: {count}')


### Bloc Type 2 — Visualisation rapide (optionnel)

Décommentez les lignes pour afficher quelques secondes du signal nettoyé. Ajustez `n_channels`, `scalings`, `tmax` ou `block` pour mieux comprendre le comportement du dataset ou illustrer un point en cours.


In [ ]:
# raw.copy().crop(tmax=5).plot(n_channels=12, scalings='auto', title='Brut prétraité (5 s)')


## 2. Préparer les événements et créer les epochs

Nous transformons les annotations textuelles en codes numériques, définissons la fenêtre temporelle des epochs et construisons un jeu d’essais équilibré (congruent vs incongruent) prêt pour les analyses N400.


### Bloc Type 1 — Extraire les événements et mapper les codes

La table `annotation_map` relie les étiquettes BIDS (`Stimulus/S 21`, etc.) aux labels lisibles (`cw_cong`, `cw_incong`).
- Adaptez la table si votre dataset contient d’autres codes.
- L’impression finale permet de vérifier le nombre d’essais disponibles par condition.


In [ ]:
# -----------------------------------------------------------------------------
# Extraction des événements et renommage en labels lisibles
# -----------------------------------------------------------------------------
annotation_map = {
    'Stimulus/S 21': 'cw_cong',
    'Stimulus/S 22': 'cw_incong',
    'Stimulus/S 31': 'question',
    'Stimulus/S 12': 'fixation',
}

events, event_id = mne.events_from_annotations(raw)
selected_event_id = {}
for original, label in annotation_map.items():
    if original in event_id:
        selected_event_id[label] = event_id[original]

if not selected_event_id:
    raise RuntimeError('Aucun événement reconnu — adaptez `annotation_map`.')

print('Étiquettes retenues:')
for label, code in selected_event_id.items():
    n_trials = int((events[:, 2] == code).sum())
    print(f' - {label:12s} → code {code:3d}, essais = {n_trials}')

focus_event_id = {label: code for label, code in selected_event_id.items() if label in FOCUS_CONDITIONS}
if len(focus_event_id) < len(FOCUS_CONDITIONS):
    print("Attention: certaines conditions d'intérêt sont absentes.")


### Bloc Type 2 — Ajuster la correspondance codes ↔ labels (optionnel)

Si vous modifiez l’encodage des événements (ex. nouvelles classes), complétez `annotation_map_custom`. C’est l’endroit idéal pour proposer des exercices étudiants (ex. ajouter la condition `question`).


In [ ]:
# annotation_map_custom = {
#     'Stimulus/S 23': 'cw_incong',
# }
# annotation_map.update(annotation_map_custom)
# print('annotation_map mis à jour :', annotation_map)


### Bloc Type 1 — Paramètres temporels (epochs)

Définissez la fenêtre d’epoch (`tmin`, `tmax`), la baseline et le critère de rejet. Pour expérimenter :
- changez la baseline (ex. `(-0.1, 0)`),
- modifiez le seuil `reject` pour observer son impact sur le nombre d’essais conservés.


In [ ]:
# -----------------------------------------------------------------------------
# Paramètres généraux d'epoching
# -----------------------------------------------------------------------------
tmin, tmax = -0.2, 0.8  # fenêtre d'epoch (en secondes)
baseline = (-0.2, 0.0)  # correction de ligne de base
reject = dict(eeg=120e-6)  # seuil de rejet automatique (120 µV)


### Bloc Type 1 — Construire les epochs centrés sur les mots cibles

Crée les epochs pour les conditions d’intérêt, les trie chronologiquement et les équilibre (`equalize_event_counts`).
- Si le dataset est déséquilibré, discutez de l’impact sur la comparaison cong/incong.
- Les impressions vous permettent de confirmer le nombre final d’epochs par condition.


In [ ]:
# -----------------------------------------------------------------------------
# Construction des epochs : congruent vs incongruent
# -----------------------------------------------------------------------------
if not focus_event_id:
    raise RuntimeError('Impossible de créer les epochs : conditions congruent/incongruent absentes.')

focus_events = [events[events[:, 2] == code] for code in focus_event_id.values()]
focus_events = np.vstack(focus_events)
focus_events = focus_events[np.argsort(focus_events[:, 0])]

epochs = mne.Epochs(
    raw,
    focus_events,
    event_id=focus_event_id,
    tmin=tmin,
    tmax=tmax,
    baseline=baseline,
    picks='eeg',
    preload=True,
    detrend=None,
    reject=reject,
    verbose='error'
)
print('Epochs brutes par condition :', {label: len(epochs[label]) for label in focus_event_id})

epochs_equalized = epochs.copy()
try:
    epochs_equalized.equalize_event_counts(focus_event_id.keys(), method='mintime')
    print('Epochs équilibrées (equalize_event_counts).')
except ValueError as exc:
    print('Equalize ignoré (', exc, ')')
    epochs_equalized = epochs

epochs = epochs_equalized


### Bloc Type 2 — Visualiser quelques epochs (optionnel)

Utilisez ces lignes (commentées) pour inspecter des epochs individuelles. Encouragez les étudiants à chercher des artefacts résiduels ou des patterns particuliers.


In [ ]:
# epochs['cw_cong'][:5].plot(n_channels=15, scalings='auto', title='Epochs congruentes (5 premières)')
# epochs['cw_incong'][:5].plot(n_channels=15, scalings='auto', title='Epochs incongruentes (5 premières)')


## 3. Construire et comparer les ERP

Nous calculons les ERP par condition, examinons leurs topographies et générons la “waveform” de différence (Incongruent – Congruent) typique d’une N400.


### Bloc Type 1 — Calculer les ERP par condition

Moyennage simple des epochs `cw_cong` et `cw_incong`. Les graphiques produits (matplotlib interactive) facilitent la discussion en classe : points d’intérêt, latence du pic négatif, etc.


In [ ]:
# -----------------------------------------------------------------------------
# Calcul des ERP pour chaque condition retenue
# -----------------------------------------------------------------------------
evokeds = {}
for label in focus_event_id:
    if len(epochs[label]) == 0:
        print(f'Avertissement: aucune epoch pour {label}.')
        continue
    evk = epochs[label].average()
    evokeds[label] = evk
    evk.plot(spatial_colors=True, time_unit='s', titles=f'ERP — {label}')


### Bloc Type 2 — Topographies temporelles (fenêtre N400)

Affiche les topomaps entre 300 et 500 ms. Vous pouvez modifier `times` pour explorer d’autres latences (P2 vers 200 ms, LPP après 500 ms) ou ajouter un curseur interactif (`mne.viz.plot_evoked_topomap`).


In [ ]:
# -----------------------------------------------------------------------------
# Topographies entre 300 et 500 ms (fenêtre N400)
# -----------------------------------------------------------------------------
if 'cw_incong' in evokeds:
    evokeds['cw_incong'].plot_topomap(
        times=np.linspace(0.30, 0.50, 6),
        ch_type='eeg',
        time_unit='s',
        colorbar=True,
        title='Topomap Incongruent (300-500 ms)'
    )
if 'cw_cong' in evokeds:
    evokeds['cw_cong'].plot_topomap(
        times=np.linspace(0.30, 0.50, 6),
        ch_type='eeg',
        time_unit='s',
        colorbar=True,
        title='Topomap Congruent (300-500 ms)'
    )


### Bloc Type 3 — Onde différence (Incongruent - Congruent)

Calcule l’onde de différence et la sauvegarde (`evoked-ave.fif`). Suggestions d’exploration :
- comparer N400 sur d’autres canaux (ex. `Pz`, `CPz`),
- tester une pondération différente (ex. moyennes pondérées si vos conditions n’ont pas le même nombre d’essais).


In [ ]:
# -----------------------------------------------------------------------------
# Calcul et sauvegarde de l'onde différence (Incongruent - Congruent)
# -----------------------------------------------------------------------------
difference = None
if {'cw_cong', 'cw_incong'}.issubset(evokeds):
    difference = mne.combine_evoked(
        [evokeds['cw_incong'], evokeds['cw_cong']],
        weights=[1, -1]
    )
    difference.plot(spatial_colors=True, time_unit='s', titles='Différence Incongruent - Congruent', gfp=True)

    session_label = session if session is not None else 'NA'
    subject_dir = (deriv_analysis / f'sub-{subject}')
    subject_dir.mkdir(parents=True, exist_ok=True)
    diff_path = subject_dir / f'sub-{subject}_ses-{session_label}_task-{TASK_LABEL}_cond-incong-minus-cong_evoked-ave.fif'
    difference.save(diff_path, overwrite=True)
    print('Onde différence sauvegardée :', diff_path)
else:
    print('Impossible de calculer la différence (ERP manquants).')


## 4. Mesures temporelles (amplitudes / latences)

Nous extrayons des métriques sur la fenêtre N400 (300-500 ms) : amplitude moyenne et latence du pic négatif. Ces mesures permettent des comparaisons statistiques ultérieures.


### Bloc Type 1 — Fonctions utilitaires

Fonctions réutilisables pour calculer la moyenne en µV et la latence du pic. Encouragez les étudiants à adapter `mode='neg'` → `'pos'` pour analyser d’autres composantes (ex. P600).


In [ ]:
# -----------------------------------------------------------------------------
# Fonctions de mesure sur les ERP
# -----------------------------------------------------------------------------
def mean_amplitude_microvolt(evoked: mne.Evoked, picks, tmin: float, tmax: float) -> dict:
    picks = picks if isinstance(picks, (list, tuple)) else [picks]
    evk = evoked.copy().pick(picks)
    start, stop = evk.time_as_index([tmin, tmax])
    if stop <= start:
        raise ValueError('Fenêtre temporelle invalide pour mean_amplitude_microvolt.')
    segment = evk.data[:, start:stop]
    return {ch: float(segment[idx].mean() * 1e6) for idx, ch in enumerate(evk.ch_names)}

def peak_latency(evoked: mne.Evoked, pick: str, tmin: float, tmax: float, mode: str = 'neg'):
    evk = evoked.copy().pick(pick)
    ch_name, time_s = evk.get_peak(tmin=tmin, tmax=tmax, mode=mode)
    time_idx = evk.time_as_index(time_s)
    amp = evk.data[0, time_idx]
    return float(time_s), float(amp * 1e6)


### Bloc Type 2 — Calculer les métriques N400 pour un sujet

Calcule et sauvegarde les métriques sujet (JSON). Incitez les étudiants à :
- tester d’autres canaux (`channel_of_interest = 'Pz'`),
- modifier la fenêtre temporelle (`time_window = (0.28, 0.45)`),
- ajouter leurs propres indicateurs (écart-type, RMS, etc.).


In [ ]:
# -----------------------------------------------------------------------------
# Calcul et sauvegarde des métriques individuelles
# -----------------------------------------------------------------------------
metrics = {}
channel_of_interest = 'Cz'
time_window = (0.30, 0.50)  # fenêtre N400

for label, evk in evokeds.items():
    if channel_of_interest not in evk.ch_names:
        continue
    amps = mean_amplitude_microvolt(evk, channel_of_interest, *time_window)
    peak_t, peak_amp = peak_latency(evk, channel_of_interest, *time_window, mode='neg')
    metrics[label] = {
        'mean_amp_uV': amps[channel_of_interest],
        'peak_time_s': peak_t,
        'peak_amp_uV': peak_amp,
    }

if difference is not None and channel_of_interest in difference.ch_names:
    diff_amp = mean_amplitude_microvolt(difference, channel_of_interest, *time_window)[channel_of_interest]
    metrics['incong_minus_cong'] = {
        'mean_amp_uV': diff_amp,
    }

session_label = session if session is not None else 'NA'
subject_dir = deriv_analysis / f'sub-{subject}'
subject_dir.mkdir(parents=True, exist_ok=True)
metrics_path = subject_dir / f'sub-{subject}_ses-{session_label}_task-{TASK_LABEL}_n400-metrics.json'
with metrics_path.open('w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
print('Métriques sauvegardées :', metrics_path)
metrics


### Bloc Type 3 — Quantification additionnelle de l'onde différence

Un exemple de calcul complémentaire sur la waveform différence. Idéal pour inviter les étudiants à implémenter leurs propres contrastes (ex. `cw_incong - question`).


In [ ]:
# -----------------------------------------------------------------------------
# Exemple : amplitude moyenne N400 sur Cz pour la différence Incongruent-Congruent
# -----------------------------------------------------------------------------
if difference is not None and channel_of_interest in difference.ch_names:
    diff_n400 = mean_amplitude_microvolt(difference, channel_of_interest, *time_window)[channel_of_interest]
    print(f'Amplitude moyenne (différence) sur {channel_of_interest} : {diff_n400:.3f} µV')
else:
    print('Aucune mesure supplémentaire disponible (onde différence manquante).')


## 5. Extension multi-sujets (optionnel)

La dernière section automatise l’extraction des ERP et des métriques pour tous les sujets. Parfait pour préparer un projet de groupe : chaque bloc montre comment constituer des “grand averages” et exporter des tableaux pour analyses statistiques.


### Bloc Type 3 — Collecter les ERP pour tous les sujets

Boucle sur les sujets détectés, construit les ERP et assemble un résumé CSV. Idées d’exercices :
- filtrer `subjects` pour tester un sous-groupe,
- comparer différentes fenêtres (`time_window`),
- ajouter des colonnes (ex. `latence pic` par sujet).


In [ ]:
# -----------------------------------------------------------------------------
# Boucle multi-sujets : stockage des ERP et calcul rapide de métriques
# -----------------------------------------------------------------------------
group_evokeds = {label: [] for label in FOCUS_CONDITIONS}
summary_rows = []

for sub in subjects:
    try:
        raw_sub, _ = load_processed_raw(sub, session=session, run=run)
    except FileNotFoundError:
        print(f'sub-{sub}: fichier prétraité manquant (ignoré).')
        continue

    events_sub, event_id_sub = mne.events_from_annotations(raw_sub)
    selected_sub = {}
    for original, label in annotation_map.items():
        if original in event_id_sub:
            selected_sub[label] = event_id_sub[original]

    focus_id_sub = {label: code for label, code in selected_sub.items() if label in FOCUS_CONDITIONS}
    if len(focus_id_sub) < len(FOCUS_CONDITIONS):
        print(f'sub-{sub}: conditions cw_cong/cw_incong incomplètes.')
        continue

    focus_events_sub = [events_sub[events_sub[:, 2] == code] for code in focus_id_sub.values()]
    focus_events_sub = np.vstack(focus_events_sub)
    focus_events_sub = focus_events_sub[np.argsort(focus_events_sub[:, 0])]

    epochs_sub = mne.Epochs(
        raw_sub,
        focus_events_sub,
        event_id=focus_id_sub,
        tmin=tmin,
        tmax=tmax,
        baseline=baseline,
        picks='eeg',
        preload=True,
        detrend=None,
        reject=reject,
        verbose='error'
    )
    try:
        epochs_sub.equalize_event_counts(focus_id_sub.keys(), method='mintime')
    except ValueError as exc:
        print(f'sub-{sub}: equalize ignoré ({exc}).')

    if any(len(epochs_sub[label]) == 0 for label in focus_id_sub):
        print(f'sub-{sub}: aucune epoch valide après filtrage.')
        continue

    subj_metrics = {}
    for label in focus_id_sub:
        evk_sub = epochs_sub[label].average()
        group_evokeds[label].append(evk_sub)
        if channel_of_interest in evk_sub.ch_names:
            subj_metrics[label] = mean_amplitude_microvolt(evk_sub, channel_of_interest, *time_window)[channel_of_interest]

    if subj_metrics:
        delta = subj_metrics.get('cw_incong', np.nan) - subj_metrics.get('cw_cong', np.nan)
        summary_rows.append((sub, subj_metrics.get('cw_cong', np.nan), subj_metrics.get('cw_incong', np.nan), delta))

print('
Résumé rapide :')
for sub, cong, incong, delta in summary_rows:
    print(f"sub-{sub}: Cong={cong:6.3f} µV  Incong={incong:6.3f} µV  Δ(Incong-Cong)={delta:6.3f} µV")

if summary_rows:
    session_label = session if session is not None else 'NA'
    group_csv = deriv_analysis / f'group_metrics_textsemantic_ses-{session_label}.csv'
    with group_csv.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['subject', 'cong_mean_uV', 'incong_mean_uV', 'delta_uV'])
        for sub, cong, incong, delta in summary_rows:
            writer.writerow([sub, f"{cong:.6f}", f"{incong:.6f}", f"{delta:.6f}"])
    print('Tableau de synthèse sauvegardé :', group_csv)
else:
    print('Aucun sujet qualifié pour la synthèse.')


### Bloc Type 1 — Calculer les grand moyennes (grand average)

Moyenne les ERP par condition et sauvegarde les `evoked-ave.fif` de groupe. Encouragez une discussion sur l’impact du nombre de sujets, du découpage, etc.


In [ ]:
# -----------------------------------------------------------------------------
# Grand moyennes (par condition) et sauvegarde
# -----------------------------------------------------------------------------
grand_averages = {}
for condition, evk_list in group_evokeds.items():
    if evk_list:
        grand_avg = mne.grand_average(evk_list)
        grand_averages[condition] = grand_avg
        print(f'{condition}: grand moyenne sur {len(evk_list)} sujets')
    else:
        print(f'{condition}: aucun ERP disponible pour le groupe.')

if grand_averages:
    group_dir = deriv_analysis / 'group'
    group_dir.mkdir(parents=True, exist_ok=True)
    for condition, grand_avg in grand_averages.items():
        group_path = group_dir / f'group_task-{TASK_LABEL}_cond-{condition}_evoked-ave.fif'
        grand_avg.save(group_path, overwrite=True)
        print('Grand average sauvegardé :', group_path)
else:
    print('Pas de grand average à sauvegarder.')


### Bloc Type 1 — Comparer les ERPs de groupe (Congruent vs Incongruent)

Superpose les grand averages sur Cz/Pz et met en évidence la fenêtre N400. Proposez aux étudiants d’ajouter un troisième subplot (ex. `CP1`) ou de changer la fenêtre pour observer d’autres effets.


In [ ]:
# -----------------------------------------------------------------------------
# Overlay des grand averages sur Cz et Pz (fenêtre N400 mise en évidence)
# -----------------------------------------------------------------------------
if {'cw_cong', 'cw_incong'}.issubset(grand_averages):
    channels_to_plot = ['Cz', 'Pz']
    n400_window = (0.30, 0.50)
    fig, axes = plt.subplots(len(channels_to_plot), 1, figsize=(12, 8), sharex=True)
    axes = np.atleast_1d(axes)

    times = grand_averages['cw_cong'].times
    for ax, ch in zip(axes, channels_to_plot):
        if ch not in grand_averages['cw_cong'].ch_names:
            ax.set_visible(False)
            continue
        cong_idx = grand_averages['cw_cong'].ch_names.index(ch)
        incong_idx = grand_averages['cw_incong'].ch_names.index(ch)
        cong = grand_averages['cw_cong'].data[cong_idx] * 1e6
        incong = grand_averages['cw_incong'].data[incong_idx] * 1e6

        ax.plot(times, cong, label='Congruent', color='tab:blue')
        ax.plot(times, incong, label='Incongruent', color='tab:red')
        ax.axvspan(*n400_window, alpha=0.15, color='gray')
        ax.axhline(0, color='black', linewidth=0.5)
        ax.axvline(0, color='black', linestyle='--', linewidth=0.8)
        ax.set_ylabel('Amplitude (µV)')
        ax.set_title(f'Grand average — {ch}')
        ax.grid(True, alpha=0.3)
        ax.legend(loc='best')

    axes[-1].set_xlabel('Temps (s)')
    fig.suptitle('Comparaison Congruent vs Incongruent (grand average)', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('Impossible de comparer : conditions de groupe manquantes.')


### Bloc Type 1 — Onde de différence de groupe (Incongruent - Congruent)

Produit et sauvegarde la waveform de groupe puis affiche les topographies de différence. Pour aller plus loin : comparer avec les ERPs fournis dans `derivatives/textsemantic-erp` ou ajouter une visualisation interactive (slider temporel).


In [ ]:
# -----------------------------------------------------------------------------
# Onde différence de groupe et topographies associées
# -----------------------------------------------------------------------------
if {'cw_cong', 'cw_incong'}.issubset(grand_averages):
    group_difference = mne.combine_evoked(
        [grand_averages['cw_incong'], grand_averages['cw_cong']],
        weights=[1, -1]
    )
    group_difference.plot(spatial_colors=True, time_unit='s', titles='Différence groupe Incongruent - Congruent', gfp=True)

    group_dir = deriv_analysis / 'group'
    diff_path = group_dir / f'group_task-{TASK_LABEL}_cond-incong-minus-cong_evoked-ave.fif'
    group_difference.save(diff_path, overwrite=True)
    print('Onde différence de groupe sauvegardée :', diff_path)

    topo_times = np.linspace(0.30, 0.50, 6)
    group_difference.plot_topomap(
        times=topo_times,
        ch_type='eeg',
        time_unit='s',
        colorbar=True,
        title='Topomap différence (300-500 ms)'
    )
else:
    print('Impossible de calculer la différence de groupe : ERP manquants.')
